# Object Recognition Using Full TensorFlow 2.x (SavedModel API)
**Full Coding available at [Object_Recognition_(Tensorflow_Edition).py](Object_Recognition_(Tensorflow_Edition).py)**
<br>
This is the documentation for Object Recognition using the **TensorFlow 2.x SavedModel API** with an SSD MobileNet V2 model loaded from TensorFlow Hub.

> **Note — TF vs TFLite comparison:**  
> | | TFLite Edition | **This (TF Edition)** |
> |---|---|---|
> | **Runtime** | `tflite_runtime.interpreter` | `tf.saved_model` via TF Hub |
> | **Model format** | `.tflite` (quantized uint8) | SavedModel (float32) |
> | **Model size** | ~4 MB | ~67 MB |
> | **Speed** | Faster (optimised for edge) | Slower but higher accuracy |
> | **Extra package** | None | `tensorflow-hub` |

***Libraries Used In The Script***
- TensorFlow (tf) — full TF2 runtime for SavedModel inference
- TensorFlow Hub (hub) — for downloading and loading the SavedModel
- OpenCV (cv2) — for image processing and display
- NumPy (np) — for numerical operations
- Picamera2 — Raspberry Pi camera interface
- libcamera — camera control library
- os / time / urllib.request — standard library utilities

## Installing Libraries & Dependencies

### Built-in Libraries
These come pre-installed with Raspberry Pi OS or Python:
- **Picamera2** — pre-installed on Raspberry Pi OS
- **libcamera** — pre-installed on Raspberry Pi OS
- **os, time, urllib.request** — built-in Python standard library

### Libraries Requiring Installation

1. **TensorFlow** [tensorflow.org](https://www.tensorflow.org/install)
    ```bash
    pip install tensorflow>=2.20.0
    ```
    Verify:
    ```python
    import tensorflow as tf
    print(tf.__version__)
    ```

2. **TensorFlow Hub** [tfhub.dev](https://www.tensorflow.org/hub)
    ```bash
    pip install tensorflow-hub
    ```
    Verify:
    ```python
    import tensorflow_hub as hub
    print(hub.__version__)
    ```
    > The model (~67 MB) is **auto-downloaded on first run** and cached in `tf_saved_model/` next to the script.

3. **OpenCV** [opencv.org](https://docs.opencv.org/4.x/d2/de6/tutorial_py_setup_in_ubuntu.html)
    ```bash
    pip install opencv-python
    ```

4. **NumPy** (usually pre-installed)
    ```bash
    pip install numpy
    ```

> **Or run `setup.sh`** from the MobileRobot directory — it installs all of the above automatically.

## Let's Start Coding!

### 1. Importing Required Libraries

This section imports all the necessary libraries:

- **`tensorflow`** — full TF2 runtime; used to build input tensors and run the SavedModel
- **`tensorflow_hub`** — downloads and loads the SSD MobileNet V2 SavedModel from TF Hub
- **`cv2`** — captures frames, resizes images, draws bounding boxes, displays results
- **`numpy`** — converts tensors to numpy arrays for post-processing
- **`Picamera2` / `libcamera`** — Raspberry Pi camera capture with autofocus
- **`os`** — file path operations and directory creation
- **`time`** — FPS calculation
- **`urllib.request`** — downloads the COCO label map on first run

In [ ]:
## Object detection using full TensorFlow 2.x (SavedModel API)
## Contrast: The TFLite edition uses tflite_runtime.interpreter (lightweight, quantized)
##           This edition uses tf.saved_model.load() (full TF2, float32, higher accuracy)
import tensorflow as tf
import tensorflow_hub as hub

## Required files validation
import os
import urllib.request

## To get the accurate time
import time

## Image acquisition and after processing
import cv2
import numpy as np
from picamera2 import Picamera2
from libcamera import controls, Transform

### 2. Global Settings and Constants

This block defines all global settings — frame resolution, model/label paths, and detection thresholds:

- **Frame resolution**: 640×480 pixels
- **`MODEL_DIR`**: Local folder where the SavedModel and label map are cached
- **`TFHUB_MODEL_URL`**: SSD MobileNet V2 (float32, 320×320) on TF Hub
- **`LABELS_URL` / `LABELS_PATH`**: COCO label map in `.pbtxt` format (auto-downloaded)
- **`SCORE_THRESHOLD`**: Minimum confidence to display a detection (default 50%)
- **`TARGET_OBJECT`**: The specific object class to trigger robot action (e.g. `'bottle'`)

```python
SCORE_THRESHOLD = 0.5
TARGET_OBJECT   = 'bottle'
```

In [ ]:
## ─── Global settings ────────────────────────────────────────────────────────
frame_height = 480
frame_width  = 640

SCRIPT_DIR   = os.path.dirname(os.path.abspath('__file__'))
MODEL_DIR    = os.path.join(SCRIPT_DIR, 'tf_saved_model')

## SSD MobileNet V2 320x320 — full TF2 SavedModel from TensorFlow Hub
## (same architecture as the TFLite edition, but full float32 precision)
TFHUB_MODEL_URL = 'https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2'

## COCO label map (90 classes, index 1-based to match model output)
LABELS_URL  = 'https://raw.githubusercontent.com/tensorflow/models/master/research/object_detection/data/mscoco_label_map.pbtxt'
LABELS_PATH = os.path.join(MODEL_DIR, 'mscoco_label_map.pbtxt')

SCORE_THRESHOLD = 0.5    # Only show detections above this confidence
TARGET_OBJECT   = 'bottle'  # The object to trigger robot action

### 3. Label Map Helpers — `ensure_labels_present()` and `load_labels_from_pbtxt()`

These two functions handle the COCO label map without requiring the `object_detection` package:

#### `ensure_labels_present()`
- Checks if `mscoco_label_map.pbtxt` exists in `MODEL_DIR`
- If missing, downloads it from the TensorFlow models GitHub repo
- Runs **once on first launch**, then reuses the cached file

```python
urllib.request.urlretrieve(LABELS_URL, LABELS_PATH)
```

#### `load_labels_from_pbtxt(path)`
- Parses the `.pbtxt` file line-by-line
- Returns a `dict {class_id: display_name}` — e.g. `{1: 'person', 44: 'bottle', ...}`
- No `label_map_util` or `object_detection` package needed

```python
labels = load_labels_from_pbtxt(LABELS_PATH)
# → {1: 'person', 2: 'bicycle', ..., 90: 'toothbrush'}
```

In [ ]:
def ensure_labels_present():
    """Download the COCO label map pbtxt if not already on disk."""
    os.makedirs(MODEL_DIR, exist_ok=True)
    if os.path.isfile(LABELS_PATH):
        return
    print('Downloading COCO label map...')
    urllib.request.urlretrieve(LABELS_URL, LABELS_PATH)
    print('Labels saved to:', LABELS_PATH)


def load_labels_from_pbtxt(path):
    """Parse a .pbtxt label map and return a dict {id: display_name}."""
    labels = {}
    current_id = None
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('id:'):
                current_id = int(line.split(':')[1].strip())
            elif line.startswith('display_name:') and current_id is not None:
                name = line.split(':', 1)[1].strip().strip('"')
                labels[current_id] = name
    return labels

### 4. Model Loading — `load_model()`

This function loads the **SSD MobileNet V2 SavedModel** from TensorFlow Hub:

- Sets `TFHUB_CACHE_DIR` to `tf_saved_model/` next to the script so the model persists across reboots (instead of using the default `/tmp` location)
- On **first run**: downloads ~67 MB from TF Hub
- On **subsequent runs**: loads instantly from the local cache

```python
os.environ.setdefault('TFHUB_CACHE_DIR', MODEL_DIR)
model = hub.load(TFHUB_MODEL_URL)
```

The model is accessed via its `serving_default` signature — this is the standard TF2 SavedModel inference endpoint:
```python
detect_fn = detector.signatures['serving_default']
```

In [ ]:
def load_model():
    """Load SSD MobileNet V2 SavedModel from TF Hub (cached locally after first run)."""
    # TF Hub caches the model in TFHUB_CACHE_DIR (default: /tmp/tfhub_modules)
    # Set cache dir next to the script for persistence across reboots
    os.environ.setdefault('TFHUB_CACHE_DIR', MODEL_DIR)
    print('Loading TF2 SavedModel from TF Hub (first run downloads ~67 MB)...')
    model = hub.load(TFHUB_MODEL_URL)
    print('Model loaded.')
    return model

### 5. Drawing Helper — `draw_detection()`

This function draws a bounding box and label directly on the BGR frame using OpenCV — no `viz_utils` package required:

- Converts **normalised coordinates** (`0.0–1.0`) from the model output to pixel coordinates
- Draws a **green rectangle** around the detected object
- Adds a **filled label bar** with the class name and confidence score

```python
# Model outputs normalised coords: ymin, xmin, ymax, xmax
left   = int(xmin * frame_width)
right  = int(xmax * frame_width)
top    = int(ymin * frame_height)
bottom = int(ymax * frame_height)
```

Returns `(left, right, top, bottom)` in pixels for computing the object centre.

In [ ]:
def draw_detection(frame, box, label, score, color=(0, 255, 0)):
    """Draw a bounding box and label on the BGR frame."""
    h, w = frame.shape[:2]
    ymin, xmin, ymax, xmax = box
    left   = int(xmin * w)
    right  = int(xmax * w)
    top    = int(ymin * h)
    bottom = int(ymax * h)

    cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
    text = f'{label}: {score:.0%}'
    text_size, _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    cv2.rectangle(frame, (left, top - text_size[1] - 4), (left + text_size[0], top), color, -1)
    cv2.putText(frame, text, (left, top - 2),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

    return left, right, top, bottom

### 6. Initialization — Labels, Model, and Camera

This block runs the setup sequence before the main loop:

1. **Download labels** (if not cached):
    ```python
    ensure_labels_present()
    labels = load_labels_from_pbtxt(LABELS_PATH)
    ```

2. **Load the TF2 SavedModel** and extract the inference function:
    ```python
    detector  = load_model()
    detect_fn = detector.signatures['serving_default']
    ```

3. **Start the camera** with RGB888 format and continuous autofocus:
    ```python
    cam = Picamera2()
    cam.configure(cam.create_preview_configuration(
        main={"format": 'RGB888', "size": (640, 480)},
        transform=Transform(vflip=1)
    ))
    cam.start()
    cam.set_controls({"AfMode": controls.AfModeEnum.Continuous})
    ```

> **Camera note:** Picamera2 `RGB888` stores pixels in **BGR order** in memory — the same layout OpenCV uses. No conversion is needed for display; only convert to RGB when feeding the TF model.

In [ ]:
## ─── Initialization ──────────────────────────────────────────────────────────

ensure_labels_present()
labels = load_labels_from_pbtxt(LABELS_PATH)

## Load the full TF2 SavedModel (infer function accepts uint8 image tensors)
detector  = load_model()
detect_fn = detector.signatures['serving_default']

## ─── Camera setup ────────────────────────────────────────────────────────────

cam = Picamera2()
cam.configure(cam.create_preview_configuration(
    main={"format": 'RGB888', "size": (frame_width, frame_height)},
    transform=Transform(vflip=1)
))
cam.start()
cam.set_controls({"AfMode": controls.AfModeEnum.Continuous})

### 7. The Main Detection Function — `object_detect()`

This function runs the real-time detection loop:

#### Frame Capture and Channel Conversion
Picamera2 `RGB888` gives **BGR bytes** (same as OpenCV), so `frame` is display-ready:
```python
frame = cam.capture_array()              # BGR — ready for OpenCV
rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # RGB — for TF inference only
```

#### Building the Input Tensor
The SavedModel expects a **uint8 RGB tensor** with a batch dimension:
```python
input_tensor = tf.convert_to_tensor(rgb)      # uint8 [H, W, 3]
input_tensor = input_tensor[tf.newaxis, ...]  # → [1, H, W, 3]
```

#### Running Inference
```python
detections = detect_fn(input_tensor)
```

#### Extracting Results
The `serving_default` output contains:
```python
boxes   = detections['detection_boxes'][0].numpy()            # [N, 4]  ymin,xmin,ymax,xmax
classes = detections['detection_classes'][0].numpy().astype(int)  # [N]
scores  = detections['detection_scores'][0].numpy()           # [N]  0.0–1.0
```

#### Processing Each Detection
For each detection above the score threshold:
- Look up the class label: `labels.get(class_id)`
- Draw bounding box with `draw_detection()`
- Compute object centre: `center_x = (left + right) // 2`
- Trigger robot action if `label == TARGET_OBJECT`

#### FPS Display and Exit
```python
cv2.imshow('Object Detection (TF2 SavedModel)', frame)  # frame is BGR ✓
if cv2.waitKey(1) & 0xFF == ord('q'):
    break
```

In [ ]:
def object_detect():
    print('Starting object detection (TF2 SavedModel). Press q to quit.')
    fps_counter = 0
    t_start = time.time()

    while True:
        ## Capture frame from camera
        # picamera2 RGB888 stores pixels in BGR order in memory — same as OpenCV
        frame = cam.capture_array()              # BGR (ready for OpenCV display)

        ## TF2 SavedModel expects uint8 RGB tensor [1, H, W, 3]
        # Convert BGR → RGB only for the inference input
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        input_tensor = tf.convert_to_tensor(rgb)             # uint8 RGB
        input_tensor = input_tensor[tf.newaxis, ...]         # add batch dim

        ## Run inference
        detections = detect_fn(input_tensor)

        ## Extract results (remove batch dimension)
        boxes   = detections['detection_boxes'][0].numpy()   # [N, 4]  ymin,xmin,ymax,xmax
        classes = detections['detection_classes'][0].numpy().astype(int)  # [N]
        scores  = detections['detection_scores'][0].numpy()  # [N]

        ## Process detections
        for i in range(len(scores)):
            if scores[i] < SCORE_THRESHOLD:
                continue

            class_id = classes[i]
            label = labels.get(class_id, f'id:{class_id}')
            score = scores[i]

            print(f'Object: {label}  Score: {score:.2f}')

            ## Draw bounding box on BGR frame (frame is already BGR)
            left, right, top, bottom = draw_detection(frame, boxes[i], label, score)

            ## Compute object centre
            center_x = (left + right) // 2
            center_y = (top + bottom) // 2
            print(f'Coordinates:  X={center_x}  Y={center_y}')

            ## Target-specific action
            if label == TARGET_OBJECT:
                coordinates = [center_x, center_y]
                object_detected = True
                # motor_command(coordinates)   # ← hook in robot movement here
            else:
                object_detected = False

        ## Compute and display FPS
        fps_counter += 1
        elapsed = time.time() - t_start
        fps = fps_counter / elapsed if elapsed > 0 else 0
        cv2.putText(frame, f'FPS: {fps:.1f}', (10, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        ## Show annotated frame (frame is BGR — correct for cv2.imshow)
        cv2.imshow('Object Detection (TF2 SavedModel)', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

### 8. Cleanup and Program Execution

#### `cleanup()`
Releases all resources cleanly — always called via `finally` regardless of how the program exits:
```python
def cleanup():
    cam.stop()
    cam.close()
    cv2.destroyAllWindows()
```

#### Entry Point with `try/finally`
Using `finally` ensures cleanup runs on **every** exit path — `q` key, `Ctrl+C`, or any unhandled exception:
```python
if __name__ == '__main__':
    try:
        object_detect()
    except KeyboardInterrupt:
        pass
    finally:
        cleanup()
```

In [ ]:
## ─── Cleanup & entry point ───────────────────────────────────────────────────

def cleanup():
    """Release all resources cleanly."""
    try:
        cam.stop()
        cam.close()
    except Exception:
        pass
    cv2.destroyAllWindows()


if __name__ == '__main__':
    try:
        object_detect()
    except KeyboardInterrupt:
        pass
    finally:
        cleanup()